# Random Forest: clasificación de la calidad global

Notebook independiente del modelo de regresión.

Se construye el **dataset de clasificación y el modelo Random Forest en este mismo notebook**, para los tres escenarios:

1. **Ayer → mañana:** información de `T-1` → calidad global de `T+1`
2. **Ayer → pasado mañana:** información de `T-1` → calidad global de `T+2`
3. **Hoy → mañana:** información de `T` → calidad global de `T+1`

La meteorología se mantiene en **T-1** en los tres escenarios.

## Calidad global

Para cada estación y día se consideran las categorías de:
- NO₂
- O₃
- PM10
- PM2.5

La calidad global es la categoría **más frecuente**.

Si hay empate, se selecciona la **categoría peor**.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()

while not (ROOT / "src" / "database" / "TFM.duckdb").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError(
            "No se ha encontrado la raíz del proyecto. "
            "Ejecuta el notebook desde el proyecto."
        )
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "modeling"

DATASETS = {
    "ayer_manana": DATA_DIR / "dataset_mag14_ayer_a_manana.parquet",
    "ayer_pasado_manana": DATA_DIR / "dataset_mag14_ayer_a_pasado_manana.parquet",
    "hoy_manana": DATA_DIR / "dataset_mag14_hoy_a_manana.parquet",
}

print("Proyecto:", ROOT)
print("Datos:", DATA_DIR)


Proyecto: c:\Users\dasab\Desktop\MASTER\TFM
Datos: c:\Users\dasab\Desktop\MASTER\TFM\data\modeling


## 1. Cargar los datasets de predictores

In [2]:
datasets = {}

for name, path in DATASETS.items():
    if not path.exists():
        raise FileNotFoundError(
            f"No existe {path}. "
            "Ejecuta primero el notebook de construcción del dataset."
        )

    d = pd.read_parquet(path)
    d["fecha_referencia"] = pd.to_datetime(d["fecha_referencia"])
    d["fecha_target"] = pd.to_datetime(d["fecha_target"])

    datasets[name] = d

    print(
        f"{name}: {d.shape}, "
        f"referencia {d['fecha_referencia'].min().date()} "
        f"→ {d['fecha_referencia'].max().date()}"
    )


ayer_manana: (23647, 39), referencia 2020-01-08 → 2024-12-30
ayer_pasado_manana: (23634, 39), referencia 2020-01-08 → 2024-12-29
hoy_manana: (23647, 39), referencia 2020-01-08 → 2024-12-30


## 2. Construcción de la calidad global

In [3]:
import duckdb

DB_PATH = ROOT / "src" / "database" / "TFM.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

quality_air = con.execute("""
    SELECT
        estacion,
        fecha,
        magnitud,
        calidad
    FROM enriched.calidad_aire_final
    WHERE magnitud IN (8, 9, 10, 14)
""").df()

con.close()

quality_air["fecha"] = pd.to_datetime(quality_air["fecha"])
quality_air["estacion"] = quality_air["estacion"].astype(str)

quality_air["calidad_norm"] = (
    quality_air["calidad"]
    .astype("string")
    .str.strip()
    .str.lower()
)

quality_wide = (
    quality_air
    .pivot_table(
        index=["estacion", "fecha"],
        columns="magnitud",
        values="calidad_norm",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={
        8: "quality_no2",
        9: "quality_o3",
        10: "quality_pm10",
        14: "quality_pm25",
    })
)

for c in [
    "quality_no2",
    "quality_o3",
    "quality_pm10",
    "quality_pm25",
]:
    if c not in quality_wide.columns:
        quality_wide[c] = pd.NA


In [4]:
# Orden de peor a mejor.
#
# Si las etiquetas de calidad de la base contienen exactamente estas categorías,
# se aplica directamente. Si existe otra etiqueta, se conserva y se coloca al
# final del orden conocido.
QUALITY_ORDER = [
    "muy mala",
    "mala",
    "regular",
    "buena",
    "muy buena",
]

QUALITY_RANK = {
    category: i
    for i, category in enumerate(QUALITY_ORDER)
}

QUALITY_COLS = [
    "quality_no2",
    "quality_o3",
    "quality_pm10",
    "quality_pm25",
]

def calcular_calidad_global(row):
    categorias = [
        row[c]
        for c in QUALITY_COLS
        if pd.notna(row[c]) and str(row[c]).strip()
    ]

    if not categorias:
        return np.nan

    categorias = [str(x).strip().lower() for x in categorias]

    frecuencias = pd.Series(categorias).value_counts()
    max_frecuencia = frecuencias.max()

    empatadas = set(
        frecuencias[frecuencias == max_frecuencia].index
    )

    # Mayor frecuencia primero.
    # En empate: categoría peor, es decir, menor posición en QUALITY_ORDER.
    return min(
        empatadas,
        key=lambda x: QUALITY_RANK.get(x, -1)
    )

quality_wide["calidad_global"] = quality_wide.apply(
    calcular_calidad_global,
    axis=1,
)

print("Distribución global:")
display(
    quality_wide["calidad_global"]
    .value_counts(dropna=False)
)


Distribución global:


calidad_global
0    20932
1    17889
2      826
3      530
4       11
5        6
Name: count, dtype: int64

## 3. Asociar la calidad global al día objetivo

In [5]:
quality_t1 = quality_wide[
    ["estacion", "fecha", "calidad_global"]
].rename(columns={
    "fecha": "fecha_target",
    "calidad_global": "target_calidad_global",
})

quality_t2 = quality_t1.rename(
    columns={"target_calidad_global": "target_calidad_global_t2"}
)

classification_datasets = {}

for name, d in datasets.items():
    cd = d.copy()

    if name in ["ayer_manana", "hoy_manana"]:
        cd = cd.merge(
            quality_t1,
            on=["estacion", "fecha_target"],
            how="left",
        )

    elif name == "ayer_pasado_manana":
        cd = cd.merge(
            quality_t2,
            on=["estacion", "fecha_target"],
            how="left",
        )
        cd["target_calidad_global"] = cd[
            "target_calidad_global_t2"
        ]
        cd = cd.drop(columns=["target_calidad_global_t2"])

    classification_datasets[name] = cd

for name, d in classification_datasets.items():
    print(f"\n{name}")
    display(
        d["target_calidad_global"]
        .value_counts(dropna=False)
    )



ayer_manana


target_calidad_global
1    13207
0     9992
2      327
3      120
5        1
Name: count, dtype: int64


ayer_pasado_manana


target_calidad_global
1    13195
0     9992
2      326
3      120
5        1
Name: count, dtype: int64


hoy_manana


target_calidad_global
1    13207
0     9992
2      327
3      120
5        1
Name: count, dtype: int64

## 4. Variables predictoras

In [6]:
TARGET = "target_calidad_global"

EXCLUDE = {
    "fecha_referencia",
    "fecha_target",
    "estacion",
    "caso",
    "target",                 # target de regresión
    TARGET,
}

def feature_columns(df):
    return [
        c for c in df.columns
        if c not in EXCLUDE
    ]

for name, d in classification_datasets.items():
    features = feature_columns(d)
    print(f"{name}: {len(features)} variables")
    print(features)


ayer_manana: 34 variables
['tipo_estacion', 'latitud', 'longitud', 'altitud', 'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_7', 'pm25_mean_3', 'pm25_mean_7', 'no2_lag_1', 'no2_lag_2', 'no2_lag_7', 'no2_mean_3', 'no2_mean_7', 'o3_lag_1', 'o3_lag_2', 'o3_lag_7', 'o3_mean_3', 'o3_mean_7', 'pm10_lag_1', 'pm10_lag_2', 'pm10_lag_7', 'pm10_mean_3', 'pm10_mean_7', 'temperatura', 'humedad', 'precipitacion', 'presion', 'viento_velocidad', 'viento_dir_sin', 'viento_dir_cos', 'laborable_target', 'mes_sin', 'mes_cos']
ayer_pasado_manana: 34 variables
['tipo_estacion', 'latitud', 'longitud', 'altitud', 'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_7', 'pm25_mean_3', 'pm25_mean_7', 'no2_lag_1', 'no2_lag_2', 'no2_lag_7', 'no2_mean_3', 'no2_mean_7', 'o3_lag_1', 'o3_lag_2', 'o3_lag_7', 'o3_mean_3', 'o3_mean_7', 'pm10_lag_1', 'pm10_lag_2', 'pm10_lag_7', 'pm10_mean_3', 'pm10_mean_7', 'temperatura', 'humedad', 'precipitacion', 'presion', 'viento_velocidad', 'viento_dir_sin', 'viento_dir_cos', 'laborable_target', 'mes_sin'

## 5. Separación temporal

Se mantiene la misma división temporal en los tres escenarios:

- **Train:** antes de 2023
- **Validation:** 2023
- **Test:** 2024 en adelante

No se realiza una división aleatoria, para evitar utilizar información futura durante el entrenamiento.


In [7]:
splits = {}

for name, d in classification_datasets.items():
    d = d.sort_values(
        ["fecha_referencia", "estacion"]
    ).reset_index(drop=True)

    train = d[
        d["fecha_referencia"] < "2023-01-01"
    ].copy()

    val = d[
        (d["fecha_referencia"] >= "2023-01-01") &
        (d["fecha_referencia"] < "2024-01-01")
    ].copy()

    test = d[
        d["fecha_referencia"] >= "2024-01-01"
    ].copy()

    train = train.dropna(subset=[TARGET])
    val = val.dropna(subset=[TARGET])
    test = test.dropna(subset=[TARGET])

    splits[name] = {
        "train": train,
        "val": val,
        "test": test,
        "all": d,
    }

    print(
        f"{name}: "
        f"train={len(train):,}, "
        f"val={len(val):,}, "
        f"test={len(test):,}"
    )


ayer_manana: train=14,157, val=4,745, test=4,745
ayer_pasado_manana: train=14,157, val=4,745, test=4,732
hoy_manana: train=14,157, val=4,745, test=4,745


## 6. Preparación de variables para Random Forest

In [8]:
def prepare_features(train, val, test, features):
    train = train.copy()
    val = val.copy()
    test = test.copy()

    categorical_cols = train[features].select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    numeric_cols = [
        c for c in features
        if c not in categorical_cols
    ]

    # Imputación numérica usando únicamente train.
    medians = train[numeric_cols].median()

    for frame in [train, val, test]:
        frame.loc[:, numeric_cols] = frame[numeric_cols].fillna(
            medians
        )

    # Variables categóricas.
    for col in categorical_cols:
        mode = train[col].mode(dropna=True)
        fill_value = mode.iloc[0] if len(mode) else "desconocido"

        for frame in [train, val, test]:
            frame.loc[:, col] = (
                frame[col]
                .fillna(fill_value)
                .astype(str)
            )

    # One-hot encoding ajustado con train.
    X_train = pd.get_dummies(
        train[features],
        columns=categorical_cols,
    )
    X_val = pd.get_dummies(
        val[features],
        columns=categorical_cols,
    )
    X_test = pd.get_dummies(
        test[features],
        columns=categorical_cols,
    )

    X_val = X_val.reindex(
        columns=X_train.columns,
        fill_value=0,
    )
    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0,
    )

    return X_train, X_val, X_test


## 7. Random Forest para los tres escenarios

In [9]:
models = {}
predictions = {}
classification_results = {}

for name, parts in splits.items():

    train = parts["train"]
    val = parts["val"]
    test = parts["test"]

    features = feature_columns(parts["all"])

    X_train, X_val, X_test = prepare_features(
        train,
        val,
        test,
        features,
    )

    encoder = LabelEncoder()

    y_train = encoder.fit_transform(
        train[TARGET]
    )
    y_val = encoder.transform(
        val[TARGET]
    )
    y_test = encoder.transform(
        test[TARGET]
    )

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)

    models[name] = {
        "model": model,
        "encoder": encoder,
        "features": X_train.columns.tolist(),
    }

    predictions[name] = {
        "val": pred_val,
        "test": pred_test,
        "y_val": y_val,
        "y_test": y_test,
        "classes": encoder.classes_,
    }

    classification_results[name] = {
        "validation": {
            "accuracy": accuracy_score(y_val, pred_val),
            "precision_macro": precision_score(
                y_val, pred_val,
                average="macro",
                zero_division=0,
            ),
            "recall_macro": recall_score(
                y_val, pred_val,
                average="macro",
                zero_division=0,
            ),
            "f1_macro": f1_score(
                y_val, pred_val,
                average="macro",
                zero_division=0,
            ),
        },
        "test": {
            "accuracy": accuracy_score(y_test, pred_test),
            "precision_macro": precision_score(
                y_test, pred_test,
                average="macro",
                zero_division=0,
            ),
            "recall_macro": recall_score(
                y_test, pred_test,
                average="macro",
                zero_division=0,
            ),
            "f1_macro": f1_score(
                y_test, pred_test,
                average="macro",
                zero_division=0,
            ),
        },
    }

    print(f"\n{name}")
    print("Clases:", list(encoder.classes_))
    print("Validation:", classification_results[name]["validation"])
    print("Test:", classification_results[name]["test"])



ayer_manana
Clases: ['0', '1', '2', '3', '5']
Validation: {'accuracy': 0.7384615384615385, 'precision_macro': 0.3884170615935632, 'recall_macro': 0.3825837527217314, 'f1_macro': 0.3797558038366862}
Test: {'accuracy': 0.7504741833508957, 'precision_macro': 0.41694573379950683, 'recall_macro': 0.403730956053765, 'f1_macro': 0.40344917603542135}

ayer_pasado_manana
Clases: ['0', '1', '2', '3', '5']
Validation: {'accuracy': 0.7342465753424657, 'precision_macro': 0.38714554771937726, 'recall_macro': 0.3933601483877918, 'f1_macro': 0.3880747688141977}
Test: {'accuracy': 0.7457734573119188, 'precision_macro': 0.4012380433057398, 'recall_macro': 0.39737661284429443, 'f1_macro': 0.3948632641936061}

hoy_manana
Clases: ['0', '1', '2', '3', '5']
Validation: {'accuracy': 0.7384615384615385, 'precision_macro': 0.3884170615935632, 'recall_macro': 0.3825837527217314, 'f1_macro': 0.3797558038366862}
Test: {'accuracy': 0.7504741833508957, 'precision_macro': 0.41694573379950683, 'recall_macro': 0.40373

## 8. Tabla de resultados

In [10]:
rows = []

for name, metrics in classification_results.items():
    for split_name, values in metrics.items():
        rows.append({
            "escenario": name,
            "split": split_name,
            **values,
        })

results_df = pd.DataFrame(rows)

display(
    results_df.sort_values(
        ["split", "f1_macro"],
        ascending=[True, False],
    )
)


,escenario,split,accuracy,precision_macro,recall_macro,f1_macro
1,ayer_manana,test,0.750474,0.416946,0.403731,0.403449
5,hoy_manana,test,0.750474,0.416946,0.403731,0.403449
3,ayer_pasado_manana,test,0.745773,0.401238,0.397377,0.394863
2,ayer_pasado_manana,validation,0.734247,0.387146,0.393360,0.388075
0,ayer_manana,validation,0.738462,0.388417,0.382584,0.379756
4,hoy_manana,validation,0.738462,0.388417,0.382584,0.379756


## 9. Matrices de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

labels = ["Muy buena", "Buena", "Regular", "Mala", "Muy mala"]

for name, data in datasets.items():
    print(name.upper())
    print("=" * 60)

    y_real = data["calidad_real"]
    y_pred = data["calidad_pred"]

    mask = y_real.notna() & y_pred.notna()

    cm = confusion_matrix(
        y_real[mask],
        y_pred[mask],
        labels=labels
    )

    display(
        pd.DataFrame(
            cm,
            index=[f"real_{x}" for x in labels],
            columns=[f"pred_{x}" for x in labels]
        )
    )


AYER_MANANA


ValueError: Shape of passed values is (4, 4), indices imply (5, 5)

## 10. Importancia de variables

In [ ]:
for name, info in models.items():

    model = info["model"]

    importance = (
        pd.DataFrame({
            "feature": info["features"],
            "importance": model.feature_importances_,
        })
        .sort_values("importance", ascending=False)
    )

    print(f"\n{name}")
    display(importance.head(20))

    top = importance.head(15).sort_values("importance")

    plt.figure(figsize=(10, 6))
    plt.barh(
        top["feature"],
        top["importance"],
    )
    plt.title(
        f"Top 15 variables - Random Forest - {name}"
    )
    plt.xlabel("Importancia")
    plt.show()


## 11. Distribución de clases por escenario

In [ ]:
for name, parts in splits.items():

    train_counts = (
        parts["train"][TARGET]
        .value_counts(normalize=True)
        .rename("train")
    )

    val_counts = (
        parts["val"][TARGET]
        .value_counts(normalize=True)
        .rename("validation")
    )

    test_counts = (
        parts["test"][TARGET]
        .value_counts(normalize=True)
        .rename("test")
    )

    distribution = pd.concat(
        [train_counts, val_counts, test_counts],
        axis=1,
    ).fillna(0)

    print(f"\n{name}")
    display(distribution)


## 12. Resumen

Este notebook es independiente del notebook de regresión.

### Clasificación
**Random Forest Classifier** → predice la **calidad global** del día objetivo.

### Tres escenarios

| Escenario | Información de calidad | Meteorología | Target |
|---|---|---|---|
| Ayer → mañana | `T-1` | `T-1` | Calidad global `T+1` |
| Ayer → pasado mañana | `T-1` | `T-1` | Calidad global `T+2` |
| Hoy → mañana | `T` | `T-1` | Calidad global `T+1` |

La calidad global se determina por mayoría entre NO₂, O₃, PM10 y PM2.5. En caso de empate se escoge la categoría peor.
